In [ ]:
import pandas as pd 
import numpy as np
# import matplotlib.pyplot as plt
import pandas as pd
import os

os.chdir("../")

In [ ]:
data = pd.read_csv("./data/raw/data.csv")
data.head()

In [ ]:
import sys
sys.path.append("src")
from common.constants import TRANSFORMED_DATA_DIR
import yaml
from dataclasses import dataclass

import os

def create_directory(dir_path):
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

@dataclass
class DataTransformationConfig:
    indir : str
    format: str
    outdir: str

@dataclass
class DataRetrievalConfig:
    persist_dir: str
    csv_file_path: str
    model: str

@dataclass
class RecommenderConfig:
    llm : str
    temperature: float


class ConfigurationManager:
    def __init__(self, config_path):
        self.config_path = config_path
        self.config = self.load_config()

    def load_config(self):
        with open(self.config_path, 'r') as file:
            config = yaml.safe_load(file)
        return config
    
    def get_data_transformation_config(self):
        return DataTransformationConfig(
            indir=self.config['data']['indir'],
            format=self.config['data']['transforms']['format'],
            outdir=TRANSFORMED_DATA_DIR
        )
    
    def get_data_retrieval_config(self,):
        return DataRetrievalConfig(
            persist_dir=self.config['data']['persist_dir'],
            csv_file_path=os.path.join(TRANSFORMED_DATA_DIR, "data.csv"),
            model=self.config['data']['embed_model']
        )
    
    def get_recommender_config(self):
        return RecommenderConfig(
            llm=self.config['recommender']['llm'],
            temperature=self.config['recommender']['temperature']
        )
    
    def __repr__(self):
        lines = [f"ConfigurationManager(config_path='{self.config_path}')"]

        def format_config(config, indent=0):
            for key, value in config.items():
                if isinstance(value, dict):
                    lines.append(' ' * indent + f"{key}:")
                    format_config(value, indent + 2)
                else:
                    lines.append(' ' * indent + f"{key}: {value}")

        format_config(self.config)
        return "\n".join(lines)

config_manager = ConfigurationManager("./config/config.yaml")

In [ ]:
class DataTransformationComponent:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def transform(self):
        data = pd.read_csv(os.path.join(self.config.indir, "data.csv"))
        create_directory(self.config.outdir)
        required_columns = {"Name", "Genres", "sypnopsis"}

        missing_columns = required_columns - set(data.columns)
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        
        transformed_csv = data[["Name", "Genres", "sypnopsis"]]
        transformed_csv['combined_info'] = transformed_csv.apply(lambda row: self.config.format.format(
            title=row['Name'],
            genres=row['Genres'],
            sypnopsis=row['sypnopsis']
        ), axis=1)  
        transformed_csv.insert(0, 'id', range(1, len(transformed_csv) + 1))
        transformed_csv[[ 'id', 'combined_info' ]].to_csv(os.path.join(self.config.outdir, "data.csv"), index=False)

    def __call__(self):
        self.transform()

data_transformation_config = config_manager.get_data_transformation_config()
DataTransformationComponent(config=data_transformation_config)()

In [ ]:
from langchain_core.prompts import PromptTemplate

def get_anime_prompt():
    template = """
You are an expert anime recommender. Your job is to help users find the perfect anime based on their preferences.

Using the following context, provide a detailed and engaging response to the user's question.

For each question, suggest exactly three anime titles. For each recommendation, include:
1. The anime title.
2. A concise plot summary (2-3 sentences).
3. A clear explanation of why this anime matches the user's preferences.

Present your recommendations in a numbered list format for easy reading.

If you don't know the answer, respond honestly by saying you don't know — do not fabricate any information.

Context:
{context}

User's question:
{question}

Your well-structured response:
"""

    return PromptTemplate(template=template, input_variables=["context", "question"])

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings


class VectorStoreComponent:
    def __init__(self, config: DataRetrievalConfig):
        self.config = config
    
    def create_vector_store(self):
        loader = CSVLoader(file_path=self.config.csv_file_path, encoding="utf-8")
        documents = loader.load()

        splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
        split_documents = splitter.split_documents(documents)

        db = Chroma.from_documents(split_documents, HuggingFaceEmbeddings(model_name=self.config.model), persist_directory=self.config.persist_dir)
        return db

    def load_vector_store(self):
        db = Chroma(persist_directory=self.config.persist_dir, embedding_function=HuggingFaceEmbeddings(model_name=self.config.model))
        retriever = db.as_retriever()
        return retriever

In [ ]:
data_retrieval_config = config_manager.get_data_retrieval_config()
vector_store_component = VectorStoreComponent(config=data_retrieval_config)
vector_store_component.create_vector_store()
retriever = vector_store_component.load_vector_store()

In [ ]:
from langchain_groq import ChatGroq
import dotenv
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

dotenv.load_dotenv()


class AnimeRecommender:
    def __init__(self, model_name: str, temperature: float):
        self.model = ChatGroq(model=model_name, temperature=temperature)
        self.chain = {"context": retriever, "question": RunnablePassthrough()} | get_anime_prompt() | self.model | StrOutputParser()
    
    def recommend(self, question: str) -> str:
        logger.info(f"Received question: {question}")
        response = self.chain.invoke(question)
        return response

In [ ]:
# Vector Store creation pipeline
from common.logger import get_logger

logger = get_logger(__name__)

class VectorStorePipeline:
    def __init__(self, config_manager: ConfigurationManager):
        self.config_manager = config_manager
    
    def run(self):
        data_transformation_config = self.config_manager.get_data_transformation_config()
        DataTransformationComponent(config=data_transformation_config)()

        data_retrieval_config = self.config_manager.get_data_retrieval_config()
        vector_store_component = VectorStoreComponent(config=data_retrieval_config)
        vector_store_component.create_vector_store()


vector_store_pipeline = VectorStorePipeline(config_manager=config_manager)
vector_store_pipeline.run()

In [ ]:
response